# Recipe 03 — Cost Tracking
> Monitor LLM token usage and estimated costs across models.

| | |
|---|---|
| **Module** | `anchor.observability` |
| **Key classes** | `CostTracker`, `CostEntry`, `CostSummary` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.observability import CostTracker, CostEntry, CostSummary

## 1 — Create a cost tracker and record usage

In [ ]:
# Define per-model pricing as (input_cost_per_1k, output_cost_per_1k)
tracker = CostTracker(model_costs={
    "claude-haiku-4-5-20251001": (0.00025, 0.00125),
    "claude-sonnet-4-20250514": (0.003, 0.015),
})
print(f"Tracker configured for {len(tracker.model_costs)} models")
for model, (inp, out) in tracker.model_costs.items():
    print(f"  {model}: ${inp}/1k input, ${out}/1k output")

In [ ]:
# Record two LLM calls
tracker.record(model="claude-haiku-4-5-20251001", input_tokens=500, output_tokens=200)
tracker.record(model="claude-sonnet-4-20250514", input_tokens=1000, output_tokens=500)

print("Recorded 2 LLM calls")

## 2 — Inspect the cost summary

In [ ]:
summary = tracker.get_summary()

print(f"Total cost     : ${summary.total_cost:.6f}")
print(f"Total input tk : {summary.total_input_tokens}")
print(f"Total output tk: {summary.total_output_tokens}")

In [ ]:
# Per-model breakdown
for model, cost in summary.by_model.items():
    print(f"  {model}: ${cost:.6f}")

In [ ]:
# Inspect individual CostEntry records
for entry in summary.entries:
    print(f"  model={entry.model}  in={entry.input_tokens}  "
          f"out={entry.output_tokens}  cost=${entry.cost:.6f}")

## 3 — Track a multi-step pipeline

In [ ]:
# Simulate a multi-step RAG pipeline
pipeline_tracker = CostTracker(model_costs={
    "claude-haiku-4-5-20251001": (0.00025, 0.00125),
    "claude-sonnet-4-20250514": (0.003, 0.015),
})

# Step 1: Query expansion (cheap model)
pipeline_tracker.record(model="claude-haiku-4-5-20251001", input_tokens=100, output_tokens=50)

# Step 2: Answer generation (powerful model)
pipeline_tracker.record(model="claude-sonnet-4-20250514", input_tokens=2000, output_tokens=800)

# Step 3: Summary (cheap model)
pipeline_tracker.record(model="claude-haiku-4-5-20251001", input_tokens=300, output_tokens=100)

result = pipeline_tracker.get_summary()
print(f"Pipeline total cost: ${result.total_cost:.6f}")
print(f"Pipeline total tokens: {result.total_input_tokens + result.total_output_tokens}")

In [ ]:
# Per-model breakdown for the pipeline
print("Per-model breakdown:")
for model, cost in result.by_model.items():
    print(f"  {model}: ${cost:.6f}")

# Show each call in order
print("\nAll entries:")
for i, entry in enumerate(result.entries):
    print(f"  Call {i+1}: {entry.model}  "
          f"in={entry.input_tokens} out={entry.output_tokens}  ${entry.cost:.6f}")

In [ ]:
# Compare cost ratio between models
haiku_cost = result.by_model.get("claude-haiku-4-5-20251001", 0)
sonnet_cost = result.by_model.get("claude-sonnet-4-20250514", 0)
total = result.total_cost

print(f"Haiku  share: {haiku_cost/total:.1%}")
print(f"Sonnet share: {sonnet_cost/total:.1%}")

## Key Takeaways
- `CostTracker` maps model names to per-1k-token pricing.
- `record()` logs each LLM call; `get_summary()` aggregates everything.
- `CostSummary` provides `total_cost`, `by_model`, and raw `entries`.
- Track costs per pipeline step to identify optimisation opportunities.

**Next:** [Metrics](04_metrics.ipynb)